# The SFT loop: fine-tune on a file of conversations

In this tutorial you fine-tune a model on a file of conversations with the cookbook's SFT pieces: a dataset, which puts the conversations in batches, the model's message formatter, which turns them into datums, and `train_sft.train`, which steps a run through the batches. Then you sample the adapter you trained.

[Chat data](201_chat_data.ipynb) explains the datums and [Operations](103_operations.ipynb) what submitting ahead is. This notebook uses the cookbook's package, `looper_cookbook`. Until that is published, run it from a checkout of the repository, in its environment. Put your API key in the settings cell's `API_KEY`.

## Settings

The model, its tokenizer, where Looper is, your project and your API key. Then the run: a LoRA's rank, the longest datum, how many conversations an optimizer step takes, how many passes over the data, and the learning rate of the first step.

In [ ]:
MODEL = "Qwen/Qwen3.5-9B"
TOKENIZER = "Qwen/Qwen3.5-9B"
BASE_URL = "https://api.looper.goaly.ai"
PROJECT = "example"
API_KEY = "YOUR_API_KEY"
LORA_RANK = 32
MAX_SEQ_LEN = 2048
BATCH_SIZE = 8
EPOCHS = 4
LEARNING_RATE = 5e-4

## Conversations in a dataset

The example file holds 16 conversations that answer a question about an animal with a three-line card: `Habitat`, `Diet` and `Fun fact`. Your own file has the same shape, one `{"messages": [...]}` per line.

`ChatDataset` puts the conversations in batches, one per optimizer step, and the formatter writes each conversation as a datum per assistant turn. Each pass over them is in its own order, which `seed` fixes. The run's step `k`, counted from 0, trains `dataset.batch(k)`, so `EPOCHS` passes are `EPOCHS * len(dataset)` steps.

In [ ]:
from importlib import resources
from uuid import uuid4

from transformers import AutoTokenizer

from looper_cookbook import train_sft
from looper_cookbook.datasets.conversations import ChatDataset, read_conversations
from looper_cookbook.message_formatters.models import message_formatter
from looper_sdk import (
    AdamW,
    Lora,
    SamplerConfig,
    SamplingParams,
    ServiceClient,
    StateCheckpoint,
    TrainingConfig,
)

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER)
formatter = message_formatter(MODEL, tokenizer, enable_thinking=False)
path = resources.files("looper_cookbook") / "example_data" / "conversations.jsonl"
dataset = ChatDataset(
    read_conversations(path), formatter, batch_size=BATCH_SIZE, max_seq_len=MAX_SEQ_LEN, seed=0
)
steps = EPOCHS * len(dataset)
print(f"{len(dataset)} batches a pass, {steps} steps. The first step's first datum trains:")
for text in formatter.targets(dataset.batch(0)[0]):
    print(repr(text))

## The run and the loop

Two configs, with two owners:

- `TrainingConfig` is the run's. It names the model, the LoRA and the optimizer, and its sizes pick the run's GPUs: `max_seq_len` is the longest datum, and `tokens_per_step` what one step trains.
- `train_sft.Config` is the loop's. A run's optimizer keeps no schedule, so the loop names each step's learning rate itself: here a linear decay from `LEARNING_RATE` to 0 over the run. `steps_ahead=1` keeps one step submitted beyond the one whose result it awaits.

In [ ]:
config = ServiceClient.Config(
    base_url=BASE_URL, project=PROJECT, api_key=API_KEY, display_name="sft-loop"
)
largest = max(sum(len(datum.tokens) for datum in dataset.batch(step)) for step in range(steps))
training = TrainingConfig(
    model=MODEL,
    kind=Lora(rank=LORA_RANK),
    tokens_per_step=max(MAX_SEQ_LEN, largest),
    max_seq_len=MAX_SEQ_LEN,
    optimizer=AdamW(learning_rate=LEARNING_RATE, betas=(0.9, 0.95), max_grad_norm=1.0),
)
loop_config = train_sft.Config(learning_rate=LEARNING_RATE, schedule="linear", steps_ahead=1)
print(f"{steps} steps, at most {largest} tokens in one")

## Train and save

`train_sft.train` takes the run, the dataset and the loop's config, and trains one optimizer step per batch. It calls `on_step` with each step's result, in order. Creating the run and saving the checkpoint stay with you.

The loss here is each step's loss sum over its target count. It should fall from one pass to the next.

In [ ]:
def report(step):
    loss = step.loss_sum / step.weighted_targets
    print(f"step {step.step:2d}/{steps}: lr {step.lr:.2e}, loss {loss:.4f}")


async def train_and_save() -> StateCheckpoint:
    async with await config.build() as service:
        print("starting the run...")
        training_client = await service.create_training_client(training, timeout_s=20 * 60)
        await train_sft.train(training_client, dataset, loop_config, steps=steps, on_step=report)

        name = f"sft-loop-{uuid4().hex[:8]}"
        saved = await (await training_client.save_state(name)).result(timeout_s=300)
        ready = await service.wait_checkpoint(saved.id, timeout_s=15 * 60)
        print(f"checkpoint {name!r}: {ready.state.value}, {ready.size_bytes} bytes")
    return saved


checkpoint = None
checkpoint = await train_and_save()

## Sample the adapter

The prompt comes from the formatter that made the datums, so it is written as the training prompts were: in the model's format, with `enable_thinking=False`. The sampler stops on the formatter's end token, and `parse_response` reads the reply back as a message. The question is about an animal the file never mentions.

In [ ]:
async def sample_checkpoint(name, question):
    params = SamplingParams(max_tokens=96, temperature=0.7, stop_token_ids=formatter.stop_token_ids)
    sampler_config = SamplerConfig(
        checkpoint_name=name, max_seq_len=MAX_SEQ_LEN, max_concurrent_sequences=4
    )
    async with await config.build() as service:
        print(f"starting a sampler of {name!r}...")
        sampler = await service.create_sampling_client(sampler_config, timeout_s=20 * 60)
        try:
            prompt = formatter.build_generation_prompt([{"role": "user", "content": question}])
            operation = await sampler.sample(prompt, params=params)
            return await operation.result(timeout_s=300)
        finally:
            await sampler.stop()


result = await sample_checkpoint(checkpoint.name, "Tell me about the puffin.")
[sequence] = result.sequences
message, _ = formatter.parse_response(sequence.token_ids)
print(message["content"])
print("finish reason:", sequence.finish_reason.value)

## What the loop does for you

`looper_cookbook/train_sft.py` is short enough to read, and meant to be copied and changed when your training needs something else. It gets four things right that a loop of your own also has to:

- **A step is its calls, then `optim_step`.** A batch too large for one call's 16 MiB payload goes over several `forward_backward` calls, whose gradients accumulate into the one step (`max_call_tokens`).
- **It submits ahead, with a limit.** `steps_ahead` steps are accepted beyond the one whose result it awaits, so the GPUs never wait for your program, and an error leaves only those few steps accepted.
- **The run's step is the clock, for the data and for the schedule.** The loop starts at the step the run is at, so a run that has trained takes only the steps left, each with its own batch and its own rate.
- **It checks each result's step** against the step it named a rate for, and raises if another writer or a rollback moved the run.

## Next steps

The same fine-tune runs as a script, with flags for the model, the data and the schedule. Without `--data` it trains a public dataset, GSM8K's math problems:

```bash
python -m looper_cookbook.recipes.sft_basic --data my_conversations.jsonl
```